In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error

In [2]:
df_epi_raw = pd.read_csv('data/epi_r.csv', sep=',')

df_epi_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 20052 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(679), str(1)
memory usage: 104.0 MB


## Duplicates


In [3]:
df_epi_raw.duplicated().sum()

np.int64(1801)

In [4]:
df_epi_drop_dup = df_epi_raw.drop_duplicates()

## DownCast 

In [5]:
(df_epi_drop_dup.nunique()).value_counts()

2        674
17736      1
8          1
1858       1
282        1
326        1
2434       1
Name: count, dtype: int64

Т.е. 674 столбцы имеют 2 униткальных значения, 1 столбец - 17736, 1 - 8 уникальных значений.

Найдем столбцы с 2 уникальными значениями.

In [6]:
mask_cols_eq2 = df_epi_drop_dup.nunique() == 2
cols_eq2 = df_epi_drop_dup.columns[mask_cols_eq2]

In [7]:
df_epi_drop_dup.columns[~mask_cols_eq2]

Index(['title', 'rating', 'calories', 'protein', 'fat', 'sodium'], dtype='str')

In [8]:
df_epi_drop_dup.info(memory_usage=True)

<class 'pandas.DataFrame'>
Index: 18251 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(679), str(1)
memory usage: 94.8 MB


In [9]:
df_epi = df_epi_drop_dup.copy()

In [10]:
for column in df_epi_drop_dup.columns[mask_cols_eq2]:
    df_epi[column] = pd.to_numeric(df_epi[column], downcast='integer')

In [11]:
df_epi.info(memory_usage=True)

<class 'pandas.DataFrame'>
Index: 18251 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(5), int8(674), str(1)
memory usage: 12.7 MB


## Columns filter

Check:
    - frequencies (kiwi not pass)
    - correlation with other columns
    - parcers (ingredient_slicer, pyfood)

In [12]:
import pandas as pd
import re


class StrictIngredientSelector:
    def __init__(self):
        # 1. Список чистых кулинарных корней (только существительные-продукты)
        # fmt: off
        food_words = [
            'almond', 'amaretto', 'anchovy', 'anise', 'apple', 'apricot', 'artichoke', 'arugula', 'pear', 'asparagus',
            'avocado', 'bacon', 'banana', 'barley', 'basil', 'bean', 'beef', 'beer', 'beet', 'berry', 'biscuit', 
            'bitters', 'blackberry', 'blueberry', 'bok choy', 'bourbon', 'bran', 'brandy', 'bread', 'breadcrumbs', 
            'brie', 'brisket', 'broccoli', 'brussel sprout', 'butter', 'buttermilk', 'squash', 'cabbage', 'cake', 
            'calvados', 'campari', 'candy', 'cantaloupe', 'capers', 'caraway', 'cardamom', 'carrot', 'cashew', 
            'cauliflower', 'caviar', 'celery', 'champagne', 'chard', 'chartreuse', 'cheddar', 'cheese', 'cherry', 
            'chestnut', 'chicken', 'chickpea', 'chile', 'chili', 'chocolate', 'cinnamon', 'citrus', 'clam', 'clove', 
            'coconut', 'cod', 'coffee', 'coriander', 'corn', 'cornmeal', 'couscous', 'crab', 'cranberry', 'cream', 
            'cucumber', 'cumin', 'currant', 'curry', 'custard', 'dairy', 'date', 'dill', 'duck', 
            'egg', 'eggplant', 'endive', 'escarole', 'fennel', 'feta', 'fig', 'fish', 'fontina', 'frangelico', 'fruit', 
            'garlic', 'ginger', 'gin', 'goat cheese', 'goose', 'gouda', 'grains', 'grape', 'grapefruit', 
            'grappa', 'green bean', 'onion', 'scallion', 'guava', 'halibut', 'ham', 'hazelnut', 
            'honey', 'honeydew', 'horseradish', 'hummus', 'ice cream', 'jalapeño', 'jam', 'jelly', 'jícama', 'kahlúa', 
            'kale', 'kiwi', 'kumquat', 'lamb', 'leek', 'legume', 'lemon', 'lemongrass', 'lentil', 'lettuce', 
            'lime', 'lingonberry', 'liqueur', 'lobster', 'lychee', 'macadamia', 'mango', 'maple syrup', 'margarita', 
            'marsala', 'marscarpone', 'marshmallow', 'martini', 'mayonnaise', 'meat', 'melon', 'mezcal', 'milk', 
            'mint', 'molasses', 'monterey jack', 'mozzarella', 'muffin', 'mushroom', 'mussel', 'mustard', 'nectarine', 
            'noodle', 'nut', 'nutmeg', 'oat', 'oatmeal', 'octopus', 'okra', 'olive', 'orange', 'oregano', 
            'orzo', 'oyster', 'pancake', 'papaya', 'paprika', 'parmesan', 'parsley', 'parsnip', 'pasta', 'pastry', 
            'pea', 'peach', 'peanut', 'pecan', 'pepper', 'pernod', 'persimmon', 'pickles', 'pine nut', 
            'pineapple', 'pistachio', 'plantain', 'plum', 'poblano', 'pomegranate', 'poppy', 'pork', 
            'port', 'potato', 'poultry', 'prosciutto', 'prune', 'pumpkin', 'punch', 'quail', 'quince', 
            'quinoa', 'rabbit', 'radicchio', 'radish', 'raisin', 'ramekin', 'raspberry', 'red wine', 'rhubarb', 
            'rice', 'ricotta', 'rosemary', 'rosé', 'rum', 'rutabaga', 'rye', 'saffron', 'sage', 'sake', 'salad', 
            'salmon', 'salsa', 'sangria', 'sardine', 'sausage', 'scallop', 'scotch', 
            'semolina', 'sesame', 'shallot', 'shellfish', 'sherry', 'shrimp', 'snapper', 'soufflé', 
            'stew', 'sour cream', 'soy', 'soy sauce', 'sparkling wine', 'spice', 'spinach', 
            'squid', 'steak', 'stock', 'strawberry', 'sweet potato', 'swiss cheese', 'taco', 
            'tamarind', 'tangerine', 'tapioca', 'tart', 'tea', 'tequila', 'thyme', 'tilapia', 'tofu', 'tomatillo', 
            'tomato', 'tortillas', 'triple sec', 'trout', 'tuna', 'turkey', 'turnip', 'vanilla', 'veal', 
            'venison', 'vermouth', 'vinegar', 'vodka', 'walnut', 'wasabi', 'watercress', 'watermelon', 
            'white wine', 'whiskey', 'wild rice', 'wine', 'yuca', 'zucchini', 
            'créme de cacao', 'yogurt', 'bulgur', 'kirsch', 'swordfish', 'chambord', 'collard greensa'
        ]
        # fmt: on

        # Компилируем регулярное выражение строго по границам целых слов (\b)
        self.food_pattern = re.compile(
            r"\b(" + "|".join(re.escape(w) for w in food_words) + r")\b",
            flags=re.IGNORECASE,
        )

        # Паттерны стоп-слов, которые превращают еду в мета-теги (например, "peanut free")
        self.meta_blacklist = re.compile(
            r"#|\d|free|low|less|added|conscious|friendly|meals|recipes|prep|required",
            flags=re.IGNORECASE,
        )

    def filter_columns(self, columns_list):
        cleaned_ingredients = []
        for col in columns_list:
            col_lower = str(col).lower().strip()

            # 1. Если строка содержит технический мусор или маркеры диет — жестко пропуeскаем
            if self.meta_blacklist.search(col_lower):
                continue

            # 2. Проверяем строгое совпадение с базой продуктов по границам слов
            if self.food_pattern.search(col_lower):
                cleaned_ingredients.append(col)

        return cleaned_ingredients


# ==========================================
# Запуск строгого конвейера
# ==========================================
all_columns_eq2 = list(df_epi.columns[mask_cols_eq2])

selector = StrictIngredientSelector()
final_ingredients = selector.filter_columns(all_columns_eq2)

print(f"Осталось чистых ингредиентов: {len(final_ingredients)}")
print("Первые элементы списка:", final_ingredients[:15])

Осталось чистых ингредиентов: 346
Первые элементы списка: ['almond', 'amaretto', 'anchovy', 'anise', 'apple', 'apple juice', 'apricot', 'artichoke', 'arugula', 'asian pear', 'asparagus', 'avocado', 'bacon', 'banana', 'barley']


In [13]:
pd.Series(list(set(all_columns_eq2) - set(final_ingredients))).to_csv('data/not_ingredient_cols.csv')

После ручного просмотра оставшихся не ингредиентов были выявлены следующие - 'marshmallow', 'créme de cacao', 'yogurt', 'bulgur', 'kirsch', 'swordfish', 'chambord', 'collard greensa'

In [14]:
pd.Series(list(set(final_ingredients))).to_csv('data/ingredient_cols.csv')

In [15]:
def titles_by_ingredients(col, df):
    return df[df[col] == 1][['title', 'rating']]
    x   
titles_by_ingredients('spirit', df_epi)

,title,rating
239,Apple Treacle Tart,4.375
420,Amaretto Olive Oil Cake,3.750
547,Blue Mountain,3.750
576,Orange Gin Fizz,0.000
769,Moulin Rouge,0.000
...,...,...
19861,Alaska,0.000
19924,Tipsy Tea with Homemade Sweet Tea,5.000
19938,Scotch Cooler,0.000
20000,Self-Starter,0.000


In [16]:
final_columns = final_ingredients + ['rating']

In [17]:
df_epi[final_columns].to_csv('data/epi_r_processed.csv', index=False)

In [18]:
assert  1===0

SyntaxError: invalid syntax (2529549948.py, line 1)